## Virtual Assisstant

We will be creating an AI-powered agent that interacts with user in diff context such as customer support, business consulting and creative content generation.<br><br>

Create an abstraction on top of LLM, to make it more adaptable and efficient for various use cases. For this:<br>
Build an Agent class that:
- Can be initialized with custom role, instructions and model parameters
- Send user input t an LLM API and process response
- Ensures LLM follows predefined roles and behaviors

In [34]:
from openai import OpenAI
from dotenv import load_dotenv
import json
import os

In [5]:
load_dotenv('config.env')
api_key = os.getenv('OPENAI_API_KEY')

In [6]:
client = OpenAI(api_key=api_key)

## Memory Layer

In [2]:
from typing import List, Dict, Literal

In [21]:
class Memory:
    def __init__(self):
        self.messages = []

    def add_message(self, role, content, tool_calls:dict=dict(), tool_call_id=None):
        message = {
            "role": role,
            "content": content,
            "tool_calls": tool_calls
        }

        if role =="tool":
            message = {
                "role": role,
                "content": content,
                "tool_call_id": tool_call_id
            }

        self.messages.append(message)
    
    def get_messages(self):
        return self.messages
    
    def last_message(self):
        if self.messages:
            return self.messages[-1]
    
    def reset(self):
        self.message = []

In [22]:
def chat_with_tools(user_query=None, memory=None, model='gpt-4o-mini', temperature=0.0, tools=None):
    messages = [{"role": "user", "content": user_query}]

    if memory:
        if user_query:
            memory.add_message(role="user", content = user_query)
        messages = memory.get_messages()

    response = client.chat.completions.create(
        model = model,
        messages = messages,
        temperature = temperature,
        tools = tools
    )

    ai_message = str(response.choices[0].message.content)
    tool_calls = response.choices[0].message.tool_calls     # model will return by deciding which tool to call
    
    if memory:
        memory.add_message(role="assistant", content = ai_message, tool_calls=tool_calls)
    
    return ai_message

## Creating Tool
- Tool object with JSON Schema

In [17]:
# define function that we want LLM to use
def power(base, exponent):
    return base ** exponent

In [26]:
tools = [{
    "type": "function",
    "function": {
        "name": "power",
        "description": "Exponention: base to the power of exponent",
        "parameters": {
            "type": "object",
            "properties": {
                "base": {"type": "number"},
                "exponent": {"type": "number"}
            },
            "required": ["base", "exponent"],
            "additionalProperties": False
        },
        "strict": True
    }
}]

In [ ]:
memory = Memory()
memory.add_message(role = 'system', content = 'You are helipful assistant')

# Call LLM with question that need tools
ai_message = chat_with_tools("2 to the power of -5?", model = 'gpt-4o', tools = tools, memory = memory)


# get the argument from tool_calls object and call the actual defined function
args = json.loads(memory.last_message()['tool_calls'][0].function.arguments)
result = power(args["base"], args["exponent"])

# get the tool_call_id and feed the LLM with the tool result
tool_call_id = memory.last_message()['tool_calls'][0].id
memory.add_message (role='tool', content = str(result), tool_call_id=tool_call_id)
ai_message = chat_with_tools(tools=tools, memory=memory)

In [40]:
memory.get_messages()

[{'role': 'system', 'content': 'You are helipful assistant', 'tool_calls': {}},
 {'role': 'user', 'content': '2 to the power of -5?', 'tool_calls': {}},
 {'role': 'assistant',
  'content': 'None',
  'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_ph9OARreLMzRGFhUiHzEQaep', function=Function(arguments='{"base":2,"exponent":-5}', name='power'), type='function')]},
 {'role': 'tool',
  'content': '0.03125',
  'tool_call_id': 'call_ph9OARreLMzRGFhUiHzEQaep'},
 {'role': 'assistant',
  'content': '\\( 2 \\) to the power of \\(-5\\) is \\( 0.03125 \\).',
  'tool_calls': None}]

## 2. Define Agent Class

**Objective**
Your task is to implement an agent that can:<br>
- Be initialized with configurable settings - name, role, instructions, model and parameters.
- Send user messages to language model, ensuring the response align wuth given role and instructions
- Return AI-generated response as a string.

**Steps**


In [18]:
class Agent:
    # create constructor with config settings
    def __init__(
        self,
        name:str = 'Agent',
        role:str = 'Personal Assistant',
        instructions:str = 'Help user with any question.',
        model:str = 'gpt-4o-mini',
        temperature:float = 0.0
    ):
        self.name = name
        self.role = role
        self.instructions = instructions
        self.model = model
        self.temperature = temperature

        self.client = OpenAI(api_key=api_key)

    # create invoke() method for agents
    def invoke(self, user_query:str) -> str:
        
        messages = [
            {
                "role": "system",
                "content": f"You're an AI Agent. Your role is {self.role}, and you need to {self.instructions}."
            },
            {
                "role": "user",
                "content": user_query
            }
        ]
        
        response = self.client.chat.completions.create(
            model = self.model,
            temperature = self.temperature,
            messages=messages
        )

        return response.choices[0].message.content

## 3. Build some agents
Below we have created 4 agents using class Agent

In [19]:
# creating a default agent
agent = Agent()

# asking simple ques
response = agent.invoke("What is the capital of France?")
print("Agent role:", agent.role)
print("Default Agent response:", response)

Agent role: Personal Assistant
Default Agent response: The capital of France is Paris.


In [ ]:
# create travel agent with role and instructions
travel_agent = Agent(
    role = 'Travel Agent',
    instructions = 'Provide travel recommendation',
    temperature=0.7
)

# ask question
travel_response = travel_agent.invoke('Which month is best to travel Jaipur?')
print("Agent role:", travel_agent.role)
print("Travel Agent response:", travel_response)

Agent role: Travel Agent
Default Agent response: The best time to visit Jaipur is between October and March. During these months, the weather is pleasant and ideal for sightseeing, with daytime temperatures ranging from 20°C to 30°C (68°F to 86°F). 

- **October to November:** This period marks the onset of winter in Jaipur, with clear skies and comfortable temperatures.
- **December to February:** These months can get a bit cooler, especially at night, but it's still a great time for outdoor activities and exploring the city's attractions.

Avoid visiting during the summer months (April to June) when temperatures can soar above 40°C (104°F), making it uncomfortable for travel. If you enjoy festivals, consider visiting in January for the Jaipur Literature Festival or in March for the colorful Holi festival.


In [23]:
# create a math tutor agent
math_tutor_agent = Agent(
    role = 'Math Tutor',
    instructions = 'Help students solve math instructions step-by-step.',
    temperature = 0.2
)

# ask question
math_tutor_response = math_tutor_agent.invoke('How to solve a quadratic equation?')
print('Agent role:', math_tutor_agent.role)
print('Math tutor response:', math_tutor_response)

Agent role: Math Tutor
Math tutor response: To solve a quadratic equation, which is typically in the form \( ax^2 + bx + c = 0 \), you can use several methods. Here are the most common methods:

### 1. Factoring
If the quadratic can be factored, you can express it as:
\[
(a_1x + b_1)(a_2x + b_2) = 0
\]
Then, set each factor equal to zero and solve for \( x \).

**Example:**
Solve \( x^2 - 5x + 6 = 0 \).

**Step 1:** Factor the quadratic.
\[
(x - 2)(x - 3) = 0
\]

**Step 2:** Set each factor to zero.
\[
x - 2 = 0 \quad \Rightarrow \quad x = 2
\]
\[
x - 3 = 0 \quad \Rightarrow \quad x = 3
\]

**Solutions:** \( x = 2 \) and \( x = 3 \).

### 2. Using the Quadratic Formula
If factoring is difficult, you can use the quadratic formula:
\[
x = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}
\]
where \( a \), \( b \), and \( c \) are the coefficients from the quadratic equation \( ax^2 + bx + c = 0 \).

**Example:**
Solve \( 2x^2 - 4x - 6 = 0 \).

**Step 1:** Identify \( a \), \( b \), and \( c \).
- \( a 

In [25]:
# create creative storyteller agent
storyteller_agent = Agent(
    role = 'Storyteller',
    instructions = 'Create imaginative stories.',
    temperature=0.9
)

story_response = storyteller_agent.invoke('Tell me a story about dragon and wizard in 10 sentence.')
print('Agent role:', storyteller_agent.role)
print('Storyteller response:', story_response)

Agent role: Storyteller
Storyteller response: In a land where the sun kissed the peaks of the Misty Mountains, there lived a wise old wizard named Eldrin. One day, while wandering through the enchanted forest, he stumbled upon a wounded dragon named Zephyr, whose scales shimmered like the night sky. Though fear gripped his heart, Eldrin saw the pain in the dragon’s eyes and decided to help. Using his magical potions, he healed Zephyr’s wing, and in gratitude, the dragon promised to grant Eldrin one wish. 

As their friendship blossomed, they discovered a dark sorceress named Morwenna, who sought to steal the dragon’s power for herself. With Zephyr’s fire and Eldrin’s spells, they confronted Morwenna in a fierce battle that shook the mountains. The two fought bravely, combining their strengths in a spectacular display of magic and might. After a harrowing struggle, they managed to outsmart the sorceress, banishing her deep into the shadows. 

In the aftermath, Eldrin wished for peace in

## 4. Add Memory Layer to your Agent

## 5. Add Self Reflection to your Agent